In [ ]:
library("Seurat")
library("tidyverse")
library('ggpubr')
library('msigdbr')
library('GSVA')
library('RColorBrewer')
library('SummarizedExperiment')

In [ ]:
figures <- "/public/home/liwang/project/lung_cancer_ST/scRNA_data/young_lung_cancer_collection/analysis_health_tumor_v6/figures"
obj <- "/public/home/liwang/project/lung_cancer_ST/scRNA_data/young_lung_cancer_collection/analysis_health_tumor_v6/objects"

In [ ]:
age_group_color <- c('Young' = '#E5368E', 'Old' = '#3C7BB0')

In [ ]:

# ltc_palettes package
main_celltype_colors <- c('T_Cell' = '#9b2226', 'NK' = '#D55D4C', 'B_Cell' = '#ca6702', 'Plasma' = '#ee9b00',
                          'Macro' = '#005F73', 'Mono' = '#0a9396', 'DC' = '#94d2bd', 'Mast' = '#66679C', 'Neutrophil' = '#e9d8a6',
                          'Endo' = '#C2C1E0', 'Tumor' = '#B2CAEE', 'Mural' = '#D17C7D', 'Fibro' = '#F3BAA5', 'Epi' = '#67ADB7', 'Imm' = '#C17F9E')

#'NK' = '#f897a1'

In [ ]:
imm_subcelltype_colors <- c(Memory_B = "#2d6037", Naive_B = "#64AE59", Plasma_B = "#839098", CCR7_CD4_Tnaive = "#ECA8A9",
                            GZMK_CD8_Tem = "#74AED4", ZNF683_CD8_Trm = "#67ADB7",CXCR6_CD4_Trm = "#E4A6BD", ISG15_Teffector = '#B3B2B3',
                            MAIT = "#F3D8E1", FOXP3_CD4_Treg = "#009170", CXCL13_CD4_Tex = "#78A040", IFITM3_CD8_Teffector = '#839098',
                            GZMB_CD8_Teffector = "#2E75AB", CXCL13_CD8_Tex = "#009393",FGFBP2_NK = "#B06E3C", XCL1_NK = "#5AA2DA",
                            NKT = "#FBD8A2", ILC = "#B84848", Mast = "#6567A0", Undetermined = "#87C3EC",
                            pDC = "#BDE6FA", cDC1 = "#D7EFFB", cDC2 = "#6EB1DE", LAMP3_DC = "#92C2DD", Neutrophil = "#4A94C6",
                            PPARG_Mono = "#FADED2", CD16_Mono = "#FAC7B3", PPARG_Macro = "#F0A29B", SPP1_Macro = "#B389B9",
                            LGMN_Macro = "#CC7892", FABP4_Macro = "#E2A2B3", CD14_Mono = "#F3C6C1", MIF_Macro = "#89558D")

In [ ]:
transparent_bg <- theme(panel.background = element_rect(fill = NA, colour = NA),
                        plot.background = element_rect(fill = NA, colour = NA),
                        legend.box.background = element_rect(fill = NA, colour = NA),
                        legend.background = element_rect(fill = NA, colour = NA))

# Bulk_Transcriptome (TCGA_LUAD, CPTAC, CDDP_EAGLE)

## Download Data

### TCGA_LUAD

In [ ]:
library('TCGAbiolinks')

In [ ]:
# GDC -> primary site: bronchus and lung -> disease type: adenomas and adenocarcinomas -> experimental: RNA-seq -> access: open -> tumor descriptor: primary

In [ ]:
proj <- "TCGA-LUAD"
query <- GDCquery(
    project = proj,
    data.category = "Transcriptome Profiling", 
    data.type = "Gene Expression Quantification",
    workflow.type = "STAR - Counts"
)

In [ ]:
query

In [ ]:
saveRDS(query, "/public/home/liwang/data/GDCdata2/TCGA_LUAD_query.rds")

In [ ]:
GDCdownload(query, directory = '/public/home/liwang/data/GDCdata2')

In [ ]:
tcga_luad <- GDCprepare(query = query, directory = '/public/home/liwang/data/GDCdata2')

In [ ]:
#save object
saveRDS(tcga_luad, paste0(obj, "/", "tcga_luad.rds"))

In [ ]:
tcga_luad <- readRDS(paste0(obj, "/", "tcga_luad.rds"))

### CPTAC

In [ ]:
# GDC -> primary site: bronchus and lung -> disease type: adenomas and adenocarcinomas -> experimental: RNA-seq -> access: open -> tumor descriptor: primary

In [ ]:
proj2 <- "CPTAC-3"
query2 <- GDCquery(
    project = proj2,
    data.category = "Transcriptome Profiling", 
    data.type = "Gene Expression Quantification",
    workflow.type = "STAR - Counts",
    access = "open",
    file.type = "tsv"
)

In [ ]:
query2

In [ ]:
saveRDS(query2, "/public/home/liwang/data/GDCdata2/CPTAC_3_query.rds")

In [ ]:
safe_GDCdownload <- function(query,
                             directory,
                             files.per.chunk = 50,
                             sleep_seconds = 60,
                             max_attempts = Inf) {
  attempt <- 1
  
  repeat {
    message("GDCdownload attempt: ", attempt)
    
    ok <- tryCatch({
      GDCdownload(
        query = query,
        directory = directory,
        files.per.chunk = files.per.chunk
      )
      TRUE
    }, error = function(e) {
      message("Download failed: ", conditionMessage(e))
      FALSE
    })
    
    if (ok) {
      message("GDCdownload finished successfully.")
      break
    }
    
    if (attempt >= max_attempts) {
      stop("GDCdownload failed after ", max_attempts, " attempts.")
    }
    
    message("Retrying after ", sleep_seconds, " seconds...")
    Sys.sleep(sleep_seconds)
    attempt <- attempt + 1
  }
}

In [ ]:
safe_GDCdownload(
    query = query2,
    directory = "/public/home/liwang/data/GDCdata2",
    files.per.chunk = 50,
    sleep_seconds = 60
)

In [ ]:
#prepare as R objects
query2_res <- getResults(query2)
dup_samples <- query2_res %>%
    count(`sample.submitter_id`) %>%
    filter(n > 1) %>%
    pull(`sample.submitter_id`)

query2_res_nodup <- query2_res %>%
    filter(!`sample.submitter_id` %in% dup_samples)

query2_no_dup <- query2
query2_no_dup$results[[1]] <- query2_res_nodup


CPTAC_3 <- GDCprepare(
    query = query2_no_dup,
    directory = "/public/home/liwang/data/GDCdata2"
  )

In [ ]:
CPTAC_3

In [ ]:
#save object
saveRDS(CPTAC_3, paste0(obj, "/", "CPTAC_3.rds"))

In [ ]:
CPTAC_3 <- readRDS(paste0(obj, "/", "CPTAC_3.rds"))

### CDDP_EAGLE-1

In [ ]:
# GDC -> primary site: bronchus and lung -> disease type: adenomas and adenocarcinomas -> experimental: RNA-seq -> access: open -> tumor descriptor: primary

In [ ]:
proj3 <- "CDDP_EAGLE-1"
query3 <- GDCquery(
    project = proj3,
    data.category = "Transcriptome Profiling", 
    data.type = "Gene Expression Quantification",
    workflow.type = "STAR - Counts",
    access = "open",
    file.type = "tsv"
)

In [ ]:
query3

In [ ]:
saveRDS(query3, "/public/home/liwang/data/GDCdata2/CDDP_EAGLE_1_query.rds")

In [ ]:
safe_GDCdownload <- function(query,
                             directory,
                             files.per.chunk = 50,
                             sleep_seconds = 60,
                             max_attempts = Inf) {
  attempt <- 1
  
  repeat {
    message("GDCdownload attempt: ", attempt)
    
    ok <- tryCatch({
      GDCdownload(
        query = query,
        directory = directory,
        files.per.chunk = files.per.chunk
      )
      TRUE
    }, error = function(e) {
      message("Download failed: ", conditionMessage(e))
      FALSE
    })
    
    if (ok) {
      message("GDCdownload finished successfully.")
      break
    }
    
    if (attempt >= max_attempts) {
      stop("GDCdownload failed after ", max_attempts, " attempts.")
    }
    
    message("Retrying after ", sleep_seconds, " seconds...")
    Sys.sleep(sleep_seconds)
    attempt <- attempt + 1
  }
}

In [ ]:
safe_GDCdownload(
    query = query3,
    directory = "/public/home/liwang/data/GDCdata2",
    files.per.chunk = 50,
    sleep_seconds = 60
)

In [ ]:
getResults(query3) %>% colnames()

In [ ]:
#prepare as R objects
query3_res <- getResults(query3)
dup_samples <- query3_res %>%
    count(`submitter_id`) %>%
    filter(n > 1) %>%
    pull(`submitter_id`)

query3_res_nodup <- query3_res %>%
    filter(!`submitter_id` %in% dup_samples)

query3_no_dup <- query3
query3_no_dup$results[[1]] <- query3_res_nodup


CDDP_EAGLE <- GDCprepare(
    query = query3_no_dup,
    directory = "/public/home/liwang/data/GDCdata2"
  )

In [ ]:
CDDP_EAGLE

In [ ]:
#save object
saveRDS(CDDP_EAGLE, paste0(obj, "/", "CDDP_EAGLE.rds"))

In [ ]:
CDDP_EAGLE <- readRDS(paste0(obj, "/", "CDDP_EAGLE.rds"))

## Preprocess Data

In [ ]:
#filter for TCGA_LUAD
TCGA_LAUD_meta <- as.data.frame(SingleCellExperiment::colData(tcga_luad))

keep <- with(TCGA_LAUD_meta,
    !is.na(sample_type) &
    sample_type == "Primary Tumor" &
    !is.na(age_at_index)
)

tcga_luad_filtered <- tcga_luad[, keep]

In [ ]:
#filter for CPTAC_3
CPTAC_3_meta <- as.data.frame(SingleCellExperiment::colData(CPTAC_3))

keep <- with(CPTAC_3_meta,
    !is.na(biospecimen_anatomic_site) &
    biospecimen_anatomic_site == "Lung" &
    !is.na(specimen_type) &
    specimen_type == "Solid Tissue" &
    !is.na(tissue_type) &
    tissue_type == "Tumor" &
    !is.na(primary_diagnosis) &
    primary_diagnosis == "Adenocarcinoma, NOS" &
    !is.na(age_at_diagnosis)
)

CPTAC_3_filtered <- CPTAC_3[, keep]

In [ ]:
#filter for CDDP_EAGLE
CDDP_EAGLE_meta <- as.data.frame(SingleCellExperiment::colData(CDDP_EAGLE))

keep <- with(CDDP_EAGLE_meta,
    !is.na(sample_type) &
    sample_type == "Primary Tumor" &
    !is.na(age_at_index)
)

CDDP_EAGLE_filtered <- CDDP_EAGLE[, keep]

In [ ]:
sum(tcga_luad_filtered$age_at_index < 41)
sum(CPTAC_3_filtered$age_at_diagnosis < 15006)
sum(CDDP_EAGLE_filtered$age_at_index < 41)

In [ ]:
#add the age_range metadata for tcga_luad (5 groups)
tcga_luad_filtered@colData$'age_range' <- cut(tcga_luad_filtered$age_at_index,
                                               breaks = c(0, 40, 50, 60, 80, Inf),
                                               labels = c('<=40', '41-50', '51-60', '61-80', '>=81'))


tcga_luad_filtered@colData$'age' <- tcga_luad_filtered@colData$'age_at_index'
tcga_luad_filtered@colData$'Dataset' <- "TCGA_LUAD"


In [ ]:
tcga_luad_filtered@colData$'age_range' %>% table()

In [ ]:
61 * 366

In [ ]:
#add the age_type metadata for CPTAC_3 (5 groups)

CPTAC_3_filtered$age_at_diagnosis <- as.numeric(CPTAC_3_filtered$age_at_diagnosis)

CPTAC_3_filtered@colData$'age_range' <- cut(CPTAC_3_filtered$age_at_diagnosis,
                                               breaks = c(0, 15006, 18666, 22326, 29280, Inf),
                                               labels =  c('<=40', '41-50', '51-60', '61-80', '>=81'))


CPTAC_3_filtered@colData$'age' <- floor(CPTAC_3_filtered@colData$'age_at_diagnosis' / 366)
CPTAC_3_filtered@colData$'Dataset' <- 'CPTAC' 
CPTAC_3_filtered@colData$'barcode' <- CPTAC_3_filtered@colData$'sample_submitter_id'


In [ ]:
#add the age_type metadata for CDDP_EAGLE (5 groups)

CDDP_EAGLE_filtered@colData$'age_range' <- cut(CDDP_EAGLE_filtered$age_at_index,
                                               breaks = c(0, 40, 50, 60, 80, Inf),
                                               labels = c('<=40', '41-50', '51-60', '61-80', '>=81'))


CDDP_EAGLE_filtered@colData$'age' <- CDDP_EAGLE_filtered@colData$'age_at_index'
CDDP_EAGLE_filtered@colData$'Dataset' <- "CDDP_EAGLE"
CDDP_EAGLE_filtered@colData$'barcode' <- CDDP_EAGLE_filtered@colData$'sample_submitter_id'

In [ ]:
CDDP_EAGLE_filtered@colData$'age_range' %>% table()

In [ ]:
#merge datasets step1

##remove_all_duplicated_genes_by_symbol
remove_all_duplicated_genes_by_symbol <- function(se, symbol_col) {
    rd <- as.data.frame(rowData(se))

    if (!symbol_col %in% colnames(rd)) {
      stop("Column not found in rowData: ", symbol_col)
    }

    gene_symbol <- as.character(rd[[symbol_col]])

    valid <- !is.na(gene_symbol) & gene_symbol != ""

    duplicated_symbol <- gene_symbol %in% gene_symbol[duplicated(gene_symbol)]

    keep <- valid & !duplicated_symbol

    se_unique <- se[keep, ]
    rownames(se_unique) <- gene_symbol[keep]

    se_unique
  }

tcga_luad_unique <- remove_all_duplicated_genes_by_symbol(
    se = tcga_luad_filtered,
    symbol_col = "gene_name"
)

CPTAC_3_unique <- remove_all_duplicated_genes_by_symbol(
    se = CPTAC_3_filtered,
    symbol_col = "gene_name"
)

CDDP_EAGLE_unique <- remove_all_duplicated_genes_by_symbol(
    se = CDDP_EAGLE_filtered,
    symbol_col = "gene_name"
)


In [ ]:
#merge dataste step2
##drop_rowRanges
drop_rowRanges <- function(se, assay_name = "unstranded") {
    SummarizedExperiment(
      assays = list(counts = assay(se, assay_name)),
      rowData = rowData(se),
      colData = colData(se),
      metadata = metadata(se)
    )
}

tcga_luad_unique_se <- drop_rowRanges(
    tcga_luad_unique,
    assay_name = "unstranded"
)

CPTAC_3_unique_se <- drop_rowRanges(
    CPTAC_3_unique,
    assay_name = "unstranded"
)

CDDP_EAGLE_unique_se <- drop_rowRanges(
    CDDP_EAGLE_unique,
    assay_name = "unstranded"
)

In [ ]:
#merge dataste step3
##common_genes
common_genes <- Reduce(
    intersect,
    list(
      rownames(tcga_luad_unique_se),
      rownames(CPTAC_3_unique_se),
      rownames(CDDP_EAGLE_unique_se) 
    )
)

tcga_luad_unique_se <- tcga_luad_unique_se[common_genes, ]
CPTAC_3_unique_se <- CPTAC_3_unique_se[common_genes, ]
CDDP_EAGLE_unique_se <- CDDP_EAGLE_unique_se[common_genes, ]

In [ ]:
#merge dataste step4
##harmonize colData and rowData
align_colData <- function(...) {
    ses <- list(...)

    all_cols <- Reduce(
      intersect,
      lapply(ses, function(se) colnames(colData(se)))
    )

    ses <- lapply(ses, function(se) {
      cd <- as.data.frame(colData(se))

      missing_cols <- setdiff(all_cols, colnames(cd))
      for (x in missing_cols) {
        cd[[x]] <- NA
      }

      cd <- cd[, all_cols, drop = FALSE]
      rownames(cd) <- colnames(se)

      colData(se) <- S4Vectors::DataFrame(cd)

      se
    })

    ses
}



ses_aligned <- align_colData(
    tcga_luad_unique_se,
    CPTAC_3_unique_se,
    CDDP_EAGLE_unique_se
)

standardize_rowData <- function(se) {
    rowData(se) <- S4Vectors::DataFrame(
      Symbol = rownames(se)
    )
    se
}

ses_aligned[[1]] <- standardize_rowData(ses_aligned[[1]])
ses_aligned[[2]] <- standardize_rowData(ses_aligned[[2]])
ses_aligned[[3]] <- standardize_rowData(ses_aligned[[3]])

LUAD_merged <- cbind(
    ses_aligned[[1]],
    ses_aligned[[2]],
    ses_aligned[[3]]
)

In [ ]:
LUAD_merged

In [ ]:
saveRDS(LUAD_merged, paste0(obj, '/LUAD_BulkRNA_merged.rds'))

In [ ]:
LUAD_merged <- readRDS(paste0(obj, '/LUAD_BulkRNA_merged.rds'))

## Signature score

### ssGSEA

In [ ]:
## cd8_tumor_reactivity_signature ('A phenotypic signature that identifies neoantigen-reactive T cells in fresh human lung cancers')
CD8_T_tumor_reactivity_signature <- c('CXCL13', 'ENTPD1', 'BATF', 'GZMB', 'CD27', 'TIGIT', 'PHLDA1',
                                    'CD74', 'HLA-DMA', 'HLA-DRA', 'HLA-DRB1', 'HLA-DPB1', 'CD3D', 'CD82',
                                    'ARL3', 'HMOX1', 'ALOX5AP', 'DUSP4', 'CARS', 'LSP1', 'CCND2',
                                    'TPI1', 'GAPDH', 'ITM2A', 'HMGN3', 'CHST12', 'NAP1L4')

## cd4_tumor_reactivity_signature ('A phenotypic signature that identifies neoantigen-reactive T cells in fresh human lung cancers')
CD4_T_tumor_reactivity_signature <- c('CXCL13', 'NR3C1', 'ADGRG1', 'NMG', 'ITM2A', 'ETV7', 'COTL1', 'B2M', 'IGFL2')


In [ ]:
# adapted from Antigen_process_and_presentation_from_KEGG
Antigen_process <- c('PSME1', 'PSME2', 'PSME3', 'TAP1', 'TAP2', 'TAPBP',
                     'B2M', 'CALR', 'CANX', 'CIITA', 'CREB1', 'CTSB', 'CTSL', 'CTSS',
                     'LGMN', 'NFYA', 'NFYB', 'NFYC', 'PDIA3', 'RFX5', 'RFXANK', 'RFXAP')

Antigen_presentation <- c('HLA-A', 'HLA-B', 'HLA-C', 'HLA-E', 'HLA-F', 'HLA-G',
                          'HLA-DRA', 'HLA-DRB5', 'HLA-DRB1', 'HLA-DQA1', 'HLA-DQB1',
                          'HLA-DQA2', 'HLA-DQB2', 'HLA-DOB', 'HLA-DMB', 'HLA-DMA',
                          'HLA-DOA', 'HLA-DPA1', 'HLA-DPB1', 'HLA-DPB2', 'HLA-DRB6')


In [ ]:
## create the signature geneset
signature_genesets <- list('TLS_signature_from_Rita_Nature_2020' = c('CD79B', 'CD1D', 'CCR6', 'LAT', 'SKAP1', 'CETP', 'EIF1AY', 'RBP5', 'PTGDS'),
                           'TLS_signature_from_Dieu_Trends_Immunol_2014' = c('CCL19', 'CCL21', 'CXCL13', 'CCR7', 'CXCR5', 'SELL', 'LAMP3'), 
                           'TLS_signature_from_Luc_Cancer_Res_2011' = c('CCL19', 'CXCL13', 'CCL21', 'IL16', 'CCL22', 'CCL17', 'ITGAL', 'ITGAD', 'ITGA4', 'ICAM3', 'VCAM1', 'MADCAM1'), 
                           'Antigen_process' = Antigen_process,
                           'Antigen_presentation' = Antigen_presentation,
                           'CD8_T_tumor_reactivity_signature' = CD8_T_tumor_reactivity_signature,
                           'CD4_T_tumor_reactivity_signature' = CD4_T_tumor_reactivity_signature
                          )



In [ ]:
## prepare the raw counts matrix 
LUAD_merged_counts <- as.data.frame(assays(LUAD_merged)[['counts']])
LUAD_merged_counts <- as.matrix(LUAD_merged_counts)

In [ ]:
# run ssgsea
LUAD_merged_ssgsea <- gsva(expr = LUAD_merged_counts, gset.idx.list = signature_genesets, method='ssgsea', kcdf='Poisson', ssgsea.norm=T)


In [ ]:
# reshape the LUAD_merged_ssgsea
LUAD_merged_ssgsea <- LUAD_merged_ssgsea %>% t() %>% as.data.frame() %>% mutate(barcode = rownames(.))

In [ ]:
is.na(LUAD_merged_ssgsea) %>% sum()

In [ ]:
# filter the patient metadata and merge the LUAD_merged_ssgsea score
LUAD_merged_ssgsea_metadata <- as.data.frame(colData(LUAD_merged)) %>%
        select(c('barcode', 'age', 'age_range', 'Dataset', 'gender', 'ajcc_pathologic_stage', 'pack_years_smoked')) %>%
        left_join(x = ., y = LUAD_merged_ssgsea, by = 'barcode')

#rownames(LUAD_merged_ssgsea_metadata) <- LUAD_merged_ssgsea_metadata[['barcode']]

In [ ]:
is.na(LUAD_merged_ssgsea_metadata) %>% sum()

### group analysis

In [ ]:
LUAD_merged_ssgsea_metadata[['age_range']] %>% unique()

In [ ]:
options(repr.plot.width =8, repr.plot.height =8)
signature_cols <- c(
    "TLS_signature_from_Rita_Nature_2020",
    "TLS_signature_from_Dieu_Trends_Immunol_2014",
    "TLS_signature_from_Luc_Cancer_Res_2011",
    "Antigen_process",
    "Antigen_presentation",
    "CD8_T_tumor_reactivity_signature",
    "CD4_T_tumor_reactivity_signature"
)

signature_labels <- c(
    "TLS_signature_from_Rita_Nature_2020" = "TLS signature (Rita Nature 2020)",
    "TLS_signature_from_Dieu_Trends_Immunol_2014" = "TLS signature (Dieu Trends Immunol 2014)",
    "TLS_signature_from_Luc_Cancer_Res_2011" = "TLS signature (Luc Cancer Res 2011)",
    "Antigen_process" = "Antigen process score",
    "Antigen_presentation" = "Antigen presentation score",
    "CD8_T_tumor_reactivity_signature" = "CD8+ T Cell Tumor Reactivity Score",
    "CD4_T_tumor_reactivity_signature" = "CD4+ T Cell Tumor Reactivity Score"
)

all_boxplots <- list()

for (sig in signature_cols) {

    plot_dat <- LUAD_merged_ssgsea_metadata %>%
      filter(!is.na(age_range), !is.na(.data[[sig]]))

    y_max <- max(plot_dat[[sig]], na.rm = TRUE)

    p <- plot_dat %>%
      ggplot(aes(x = age_range, y = .data[[sig]], fill = age_range)) +
      stat_boxplot(
        geom = "errorbar",
        width = 0.5,
        linewidth = 1,
        position = position_dodge(0.9)
      ) +
      geom_boxplot(
        notch = FALSE,
        linewidth = 1,
        width = 0.8,
        position = position_dodge(0.9)
      ) +
      stat_compare_means(
        method = "kruskal.test",
        label = "p.format",
        label.x = 3,
        label.y = y_max * 1.1,
        size = 5.88
      ) +
      scale_fill_manual(
        values = c(
          "<=40" = "#FF9DA0",
          "41-50" = "#EF856A",
          "51-60" = "#4A94C6",  
          "61-80" = "#107AAC",
          ">=81" = "#1864AA"
        )
      ) +
      xlab(NULL) +
      ylab(signature_labels[[sig]]) +
      guides(fill = guide_legend(
        title = "Age Range",
        title.theme = element_text(size = 20)
      )) +
      theme_classic(base_size = 25) +
      theme(
        legend.position = "none",
        axis.text = element_text(colour = "black"),
        axis.title.x = element_text(margin = margin(15, 0, 0, 0)),
        axis.title.y = element_text(margin = margin(0, 15, 0, 0))
      )

    all_boxplots[[sig]] <- p
    print(p)
    ggsave(paste0(figures, '/Figures_raw/TCGA_CPTAC_CDDP_', sig, '_AgeGroup_BoxPlot.pdf'), device = 'pdf', width = 6, height = 6, bg = 'transparent' )

}


### cutoff sensitivity analysis

In [ ]:
options(repr.plot.width =10, repr.plot.height =5)
#
signature_cols <- c(
    "TLS_signature_from_Rita_Nature_2020",
    "TLS_signature_from_Dieu_Trends_Immunol_2014",
    "TLS_signature_from_Luc_Cancer_Res_2011",
    "Antigen_process",
    "Antigen_presentation",
    "CD8_T_tumor_reactivity_signature",
    "CD4_T_tumor_reactivity_signature"
  )

#
  comparison_df <- tibble(
    comparison = c(
      "<=40 vs 60-80",
      "<=45 vs 60-80",
      "<=50 vs 60-80"
    ),
    young_cutoff = c(40, 45, 50),
    old_lower = c(60, 60, 60),
    old_upper = c(80, 80, 80)
  )

  #计算 effect size、bootstrap CI、P value：

  set.seed(123)
#
  bootstrap_median_diff <- function(y, x, n_boot = 2000) {
    boot_vals <- replicate(n_boot, {
      y_boot <- sample(y, size = length(y), replace = TRUE)
      x_boot <- sample(x, size = length(x), replace = TRUE)

      median(y_boot, na.rm = TRUE) - median(x_boot, na.rm = TRUE)
    })

    quantile(boot_vals, probs = c(0.025, 0.975), na.rm = TRUE)
  }
#
  run_one_comparison <- function(comparison, young_cutoff, old_lower, old_upper) {
    young_scores <- dat_forest %>%
      filter(age <= young_cutoff) %>%
      pull(signature_score)

    old_scores <- dat_forest %>%
      filter(age >= old_lower, age <= old_upper) %>%
      pull(signature_score)

    if (length(young_scores) < 3 || length(old_scores) < 3) {
      return(tibble(
        comparison = comparison,
        young_cutoff = young_cutoff,
        old_lower = old_lower,
        old_upper = old_upper,
        n_young = length(young_scores),
        n_old = length(old_scores),
        median_young = NA_real_,
        median_old = NA_real_,
        median_difference = NA_real_,
        ci_lower = NA_real_,
        ci_upper = NA_real_,
        p_value = NA_real_
      ))
    }

    ci <- bootstrap_median_diff(young_scores, old_scores)

    tibble(
      comparison = comparison,
      young_cutoff = young_cutoff,
      old_lower = old_lower,
      old_upper = old_upper,
      n_young = length(young_scores),
      n_old = length(old_scores),
      median_young = median(young_scores, na.rm = TRUE),
      median_old = median(old_scores, na.rm = TRUE),
      median_difference = median(young_scores, na.rm = TRUE) -
        median(old_scores, na.rm = TRUE),
      ci_lower = ci[1],
      ci_upper = ci[2],
      p_value = wilcox.test(young_scores, old_scores, exact = FALSE)$p.value
    )
  }

  dat <- LUAD_merged_ssgsea_metadata
  all_forest_res <- list()
  all_forest_plots <- list()

  for (sig in signature_cols) {

    dat_forest <- dat %>%
      select(
        barcode,
        age,
        Dataset,
        signature_score = all_of(sig)
      ) %>%
      mutate(
        age = as.numeric(age),
        signature_score = as.numeric(signature_score)
      ) %>%
      filter(!is.na(age), !is.na(signature_score))

    forest_res <- pmap_dfr(comparison_df, run_one_comparison) %>%
      mutate(signature = sig)

    all_forest_res[[sig]] <- forest_res

    forest_plot_dat <- forest_res %>%
      mutate(
        comparison = factor(comparison, levels = rev(comparison_df$comparison)),
        p_label = case_when(
          is.na(p_value) ~ "NA",
          p_value < 0.001 ~ "P < 0.001",
          TRUE ~ paste0("P = ", signif(p_value, 2))
        ),
        n_label = paste0(n_young, " / ", n_old)
      )

    x_range <- range(c(forest_plot_dat$ci_lower, forest_plot_dat$ci_upper), na.rm = TRUE)
    x_pad <- diff(x_range) * 0.4

    p_forest <- ggplot(forest_plot_dat, aes(x = median_difference, y = comparison)) +
      geom_vline(
        xintercept = 0,
        linetype = "dashed",
        linewidth = 0.4,
        color = "grey40"
      ) +
      geom_errorbarh(
        aes(xmin = ci_lower, xmax = ci_upper),
        height = 0.18,
        linewidth = 0.7
      ) +
      geom_point(size = 3) +
      geom_text(
        aes(x = x_range[2] + x_pad * 0.15, label = n_label),
        hjust = 0,
        size = 3.5
      ) +
      geom_text(
        aes(x = x_range[2] + x_pad * 0.68, label = p_label),
        hjust = 0,
        size = 3.5
      ) +
      coord_cartesian(
        xlim = c(x_range[1] - x_pad * 0.15, x_range[2] + x_pad * 1.25),
        clip = "off"
      ) +
      theme_classic(base_size = 14) +
      theme(
        plot.margin = margin(5.5, 120, 5.5, 5.5),
        axis.title.y = element_blank()
      ) +
      labs(
        x = "Median difference in signature score (young - old)",
        title = paste0("Sensitivity of young-old differences: ", sig)
      )

    all_forest_plots[[sig]] <- p_forest

    print(p_forest)

    ggsave(paste0(figures, '/Figures_raw/TCGA_CPTAC_CDDP_', sig, '_AgeCutoffAnalysis_ForestPlot.pdf'), device = 'pdf', width = 8, height = 4, bg = 'transparent' )
  }